### Complete HT table results  
We would like to include **p-values, overlap rate, intensity and confidence scores**.   
Allow easy adding cell types and qualified sections.

In [1]:
source("./scale.R")
test <- read.csv("/data/coro_data.csv")
test = test %>% rename(x = x_section, y = y_section)
options(warn=-1)

Loading required package: spatstat.data

Loading required package: spatstat.univar

spatstat.univar 3.1-1

Loading required package: spatstat.geom

spatstat.geom 3.3-4

Loading required package: spatstat.random

spatstat.random 3.3-2

Loading required package: spatstat.explore

Loading required package: nlme

spatstat.explore 3.3-3

Loading required package: spatstat.model

Loading required package: rpart

spatstat.model 3.3-3

Loading required package: spatstat.linnet

spatstat.linnet 3.2-3


spatstat 3.3-0 
For an introduction to spatstat, type ‘beginner’ 



Attaching package: ‘dplyr’


The following object is masked from ‘package:nlme’:

    collapse


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
seeds <- sample(1:100,5,replace=FALSE)
seeds

[1] 25 42 69 17 61

In [3]:
library('foreach')
library('doParallel')
cores=detectCores()
cl <- makeCluster(cores[1]-1) #not to overload your computer
registerDoParallel(cl)

Loading required package: iterators

Loading required package: parallel



## HT table calculation

In [4]:
ecdf_fun <- function(x,perc) ecdf(x)(perc)
## Calculate column stats for each type
calc_pp_2d<-function(typ,pp,ps,section){
    nnp <- c()
    nni <- c()
    mnn <- c()
    res<- list()
    for (i in 1:length(pp)){     
        perc<- ecdf_fun(nndist(pp[[i]]), 0.01)
        int<- intensity(pp[[i]])
        mval<- mean(marks(pp[[i]]))
        nnp<-append(nnp,perc)
        nni<-append(nni,int)
        mnn <- append(mnn, mval)
        res[[as.character(section[i])]] = min(ps[,i])    
    }   
    pval <- signif(fisher(unname(unlist(res)))$p,2)
    res[['pval']] = pval
    res[['perc']] = round(mean(nnp)*100,2)
    res[['lam']] = round(mean(nni))
    res[['conf']] = round(mean(mnn),2)
    res[['rm']] = length(ps[,1])
    res[['Cluster']] = typ
    return(res)
}

### Inhibitory clusters and supertypes

In [5]:
df<- read.csv("filter-inhib.csv")
df1 <- replace(df, is.na(df), 0)
df1[df1<=0.5] <- NA
df1<-df1[rowSums(is.na(df1)) != ncol(df1)-1, ]
row.names(df1) <- NULL
types = df1$cluster
types

[1] "0740 Pvalb Gaba_3" "0742 Pvalb Gaba_3" "0776 Sst Gaba_4"  
[4] "0798 Sst Gaba_9"   "0819 Sst Gaba_16"

In [6]:
sections<- apply(df1[,-1], 1, function(i) colnames(df1[,-1])[ !is.na(i) ]) ## qualified colns
sections<- sapply(sections, function(x){as.numeric(gsub("X.", "", x))})

In [7]:
df = data.frame(matrix(vector(), 0, 9,
                dimnames=list(c(), c('Cluster','59','60','61',"pval", "perc", "lam",'conf','rm'))),
                stringsAsFactors=F)
df$Cluster <- as.character(df$Cluster)
df

Cluster,X59,X60,X61,pval,perc,lam,conf,rm
<chr>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>


In [8]:
set.seed(seeds[1]) 
res1 <- foreach(t = 0:length(types), .combine=function(x,y) bind_rows(as.data.frame(x),as.data.frame(y)),.packages=c('dplyr','spatstat','poolr')) %dopar% {
    if (t == 0){
        q <- df
    }
    else{
        pp<-create_pp(test, NULL,sections[[t]], types[t])
        rm<-min(as.numeric(quantile(do.call(c,lapply(pp, nndist)),0.5)),0.12) 
        ps<-get_pp_2d(pp, rm, FALSE)
        
        ## write into data frame
        q <- calc_pp_2d(types[t],pp,ps,sections[[t]])    
        q
    }
}
res1

Cluster,X59,X60,X61,pval,perc,lam,conf,rm
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
0740 Pvalb Gaba_3,NA,NA,0.005,0.005,0,123,0.71,55
0742 Pvalb Gaba_3,NA,NA,0.005,0.005,0,63,0.71,72
0776 Sst Gaba_4,NA,0.01,NA,0.010,0,18,0.71,111
0798 Sst Gaba_9,0.03,NA,NA,0.030,0,13,0.70,111
0819 Sst Gaba_16,0.01,NA,NA,0.010,0,9,0.69,111


In [9]:
write.csv(res1, "table-in1.csv",row.names = FALSE)

In [10]:
set.seed(seeds[2]) 
res2 <- foreach(t = 0:length(types), .combine=function(x,y) bind_rows(as.data.frame(x),as.data.frame(y)),.packages=c('dplyr','spatstat','poolr')) %dopar% {
    if (t == 0){
        q <- df
    }
    else{
        pp<-create_pp(test, NULL,sections[[t]], types[t])
        rm<-min(as.numeric(quantile(do.call(c,lapply(pp, nndist)),0.5)),0.12) 
        ps<-get_pp_2d(pp, rm, FALSE)
        ## write into data frame
        q <- calc_pp_2d(types[t],pp,ps,sections[[t]])    
        q
    }
}
res2

Cluster,X59,X60,X61,pval,perc,lam,conf,rm
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
0740 Pvalb Gaba_3,NA,NA,0.005,0.005,0,123,0.71,55
0742 Pvalb Gaba_3,NA,NA,0.005,0.005,0,63,0.71,72
0776 Sst Gaba_4,NA,0.01,NA,0.010,0,18,0.71,111
0798 Sst Gaba_9,0.095,NA,NA,0.095,0,13,0.70,111
0819 Sst Gaba_16,0.025,NA,NA,0.025,0,9,0.69,111


In [11]:
write.csv(res2, "table-in2.csv",row.names = FALSE)

In [12]:
set.seed(seeds[3]) 
res3 <- foreach(t = 0:length(types), .combine=function(x,y) bind_rows(as.data.frame(x),as.data.frame(y)),.packages=c('dplyr','spatstat','poolr')) %dopar% {
    if (t == 0){
        q <- df
    }
    else{
        pp<-create_pp(test, NULL,sections[[t]], types[t])
        rm<-min(as.numeric(quantile(do.call(c,lapply(pp, nndist)),0.5)),0.12) 
        ps<-get_pp_2d(pp, rm, FALSE)
        ## write into data frame
        q <- calc_pp_2d(types[t],pp,ps,sections[[t]])    
        q
    }
}
res3

Cluster,X59,X60,X61,pval,perc,lam,conf,rm
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
0740 Pvalb Gaba_3,NA,NA,0.005,0.005,0,123,0.71,55
0742 Pvalb Gaba_3,NA,NA,0.005,0.005,0,63,0.71,72
0776 Sst Gaba_4,NA,0.01,NA,0.010,0,18,0.71,111
0798 Sst Gaba_9,0.03,NA,NA,0.030,0,13,0.70,111
0819 Sst Gaba_16,0.03,NA,NA,0.030,0,9,0.69,111


In [13]:
write.csv(res3, "table-in3.csv",row.names = FALSE)

In [14]:
set.seed(seeds[4]) 
res4 <- foreach(t = 0:length(types), .combine=function(x,y) bind_rows(as.data.frame(x),as.data.frame(y)),.packages=c('dplyr','spatstat','poolr')) %dopar% {
    if (t == 0){
        q <- df
    }
    else{
        pp<-create_pp(test, NULL,sections[[t]], types[t])
        rm<-min(as.numeric(quantile(do.call(c,lapply(pp, nndist)),0.5)),0.12) 
        
        ps<-get_pp_2d(pp, rm, FALSE)
        ## write into data frame
        q <- calc_pp_2d(types[t],pp,ps,sections[[t]])    
        q
    }
}
res4

Cluster,X59,X60,X61,pval,perc,lam,conf,rm
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
0740 Pvalb Gaba_3,NA,NA,0.005,0.005,0,123,0.71,55
0742 Pvalb Gaba_3,NA,NA,0.005,0.005,0,63,0.71,72
0776 Sst Gaba_4,NA,0.01,NA,0.010,0,18,0.71,111
0798 Sst Gaba_9,0.020,NA,NA,0.020,0,13,0.70,111
0819 Sst Gaba_16,0.015,NA,NA,0.015,0,9,0.69,111


In [15]:
write.csv(res4, "table-in4.csv",row.names = FALSE)

In [16]:
set.seed(seeds[5]) 
res5 <- foreach(t = 0:length(types), .combine=function(x,y) bind_rows(as.data.frame(x),as.data.frame(y)),.packages=c('dplyr','spatstat','poolr')) %dopar% {
    if (t == 0){
        q <- df
    }
    else{
        pp<-create_pp(test, NULL,sections[[t]], types[t])
        rm<-min(as.numeric(quantile(do.call(c,lapply(pp, nndist)),0.5)),0.12) 
        
        ps<-get_pp_2d(pp, rm, FALSE)
        ## write into data frame
        q <- calc_pp_2d(types[t],pp,ps,sections[[t]])    
        q
    }
}
res5

Cluster,X59,X60,X61,pval,perc,lam,conf,rm
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
0740 Pvalb Gaba_3,NA,NA,0.005,0.005,0,123,0.71,55
0742 Pvalb Gaba_3,NA,NA,0.005,0.005,0,63,0.71,72
0776 Sst Gaba_4,NA,0.01,NA,0.010,0,18,0.71,111
0798 Sst Gaba_9,0.030,NA,NA,0.030,0,13,0.70,111
0819 Sst Gaba_16,0.015,NA,NA,0.015,0,9,0.69,111


In [17]:
write.csv(res5, "table-in5.csv",row.names = FALSE)

In [ ]:
res1<-read.csv("./outs/table-in1.csv")
res2<-read.csv("./outs/table-in2.csv")
res3<-read.csv("./outs/table-in3.csv")
res4<-read.csv("./outs/table-in4.csv")
res5<-read.csv("./outs/table-in5.csv")

In [20]:
res<-Reduce("+", list(res1[,2:4],res2[,2:4],res3[,2:4],res4[,2:4],res5[,2:4])) / 5
res<-data.frame(Cluster=res1$Cluster, res)

pv = c()
for(i in 1:nrow(res)) {
    row <- na.omit(unlist(unname(res[i,2:4])))
    pval <- signif(fisher(row)$p,2)
    pv<-append(pv,pval)
}
res$pval<-pv
res$perc<-res1$perc
res$lam<-res1$lam
res$conf<-res1$conf
res$rm<-res1$rm
res

Cluster,X59,X60,X61,pval,perc,lam,conf,rm
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
0740 Pvalb Gaba_3,NA,NA,0.005,0.005,0,123,0.71,55
0742 Pvalb Gaba_3,NA,NA,0.005,0.005,0,63,0.71,72
0776 Sst Gaba_4,NA,0.01,NA,0.010,0,18,0.71,111
0798 Sst Gaba_9,0.041,NA,NA,0.041,0,13,0.70,111
0819 Sst Gaba_16,0.019,NA,NA,0.019,0,9,0.69,111


In [21]:
write.csv(res, "table-in.csv",row.names = FALSE)